# Capa Silver - Catálogo de Empleados

## Librerias

In [1]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window

## Variables

In [2]:
%run ../00-Modulos/n_modulo_variables.ipynb

Servicios:
 Warehouse  = s3a://iceberg/warehouse
 EndPoint   = http://localhost:9000
 Uri        = thrift://localhost:9083
Base de Datos Landing =  iceberg.landing
Base de Datos Bronze  =  iceberg.bronze
Base de Datos Silver  =  iceberg.silver
Base de Datos Gold    =  iceberg.gold


## Spark Session

In [3]:
spark = spark_session_hive("Lakehouse-Iceberg")

## Extracción

In [4]:
df_employee = spark.read.table(f'{vSchemaBron}.employee')
df_salaries = spark.read.table(f'{vSchemaBron}.salaries')

## Transformaciones

In [5]:
# Filtro de empleados validos
df_filter = df_employee.where(f.col("EMPLOYEE_ID").isNotNull())

# Filtro de empleados unicos
df_filter = df_filter.withColumn("ROW_NUM", f.row_number().over(Window.partitionBy(f.col("EMPLOYEE_ID")).orderBy(f.col("DATE_ENTERED").desc())))
df_employee_unique = df_filter.where(f.col("ROW_NUM") == 1)

# Generación de columnas calculadas
df_employee_unique = df_employee_unique \
    .withColumn("EMPLOYEE_NAME", combinar_nombre_apellido_udf(f.col("FIRST_NAME"), f.col("LAST_NAME"))) \
    .withColumn("EMPLOYEE_PHONE_NMBR", formato_telefono_udf(f.col("PHONE_NUMBER"))) \
    .withColumn("EMPLOYEE_GENDER", genero_completo_udf(f.col("GENDER"))) \
    .withColumn("AGE_GROUP", grupo_edad_udf(f.col("AGE")))  

# Seleccion de gerentes
df_employee_manager = df_employee_unique.select(
    f.col("DEALERSHIP_ID"),
    f.col("EMPLOYEE_NAME").alias("DEALERSHIP_MANAGER")
).where(f.col("POSITION_TYPE") == 'MANAGER')

# Seleccion de columnas finales
df_source = df_employee_unique.alias("e") \
    .join(df_employee_manager.alias("m"), (f.col("e.DEALERSHIP_ID") == f.col("m.DEALERSHIP_ID")), "inner") \
    .join(df_salaries.alias("s"), (f.col("e.EMPLOYEE_ID") == f.col("s.EMPLOYEE_ID")), "left") \
    .select(
        f.col("e.EMPLOYEE_ID"),
        f.col("e.EMPLOYEE_NAME"),
        f.col("e.ADDRESS").alias("EMPLOYEE_ADDRESS"),
        f.col("e.CITY").alias("EMPLOYEE_CITY"),
        f.col("e.STATE").alias("EMPLOYEE_STATE"),
        f.col("e.ZIP_CODE").alias("EMPLOYEE_ZIP_CODE"),
        f.col("e.COUNTRY").alias("EMPLOYEE_COUNTRY"),
        f.col("e.EMPLOYEE_PHONE_NMBR"),
        f.col("e.FAX_NUMBER").alias("EMPLOYEE_FAX_NMBR"),
        f.col("e.EMAIL").alias("EMPLOYEE_EMAIL"),
        f.col("e.EMPLOYEE_GENDER"),
        f.col("s.SALARY").alias("EMPLOYEE_SALARY"),
        f.col("e.AGE_GROUP"),
        f.col("e.NATIVE_LANGUAGE").alias("NATIVE_LANG_DESC"),
        f.col("e.SECOND_LANGUAGE").alias("SEC_LANG_DESC"),
        f.col("e.THIRD_LANGUAGE").alias("TER_LANG_DESC"),
        f.col("e.POSITION_TYPE"),
        f.col("e.DEALERSHIP_ID"),
        f.col("e.REGIONAL_MANAGER"),
        f.col("m.DEALERSHIP_MANAGER"),
        f.col("e.HIRE_DATE"),
        f.col("e.DATE_ENTERED")
    )

## Carga

In [6]:
# Obtenemos variables para merge
target_table = f'{vSchemaSilv}.employees'
join_columns = ["EMPLOYEE_ID"]

# Realiza el merge de la información
iceberg_upsert(df_source, target_table, join_columns)

## Cierre de SparkSession

In [7]:
stop_session(spark)

Sesión de Spark cerrada correctamente.
